In [1]:
import os
import json

import pandas as pd
import torch
from datasets import Dataset
from modelscope import snapshot_download,AutoTokenizer
# from swanlab.integration.huggingface import SwanLabCallback
from peft import LoraConfig,TaskType, get_peft_model
from transformers import AutoModelForCausalLM,TrainingArguments,Trainer,DataCollatorForSeq2Seq
# import swanlab

/Users/bowie/anaconda3/envs/good/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 一 配置参数

In [12]:
class Arguments(object):
    file_path = 'ccfbdci.jsonl'
    train_file_path = 'train.jsonl'

argument = Arguments()

## 二 数据处理

In [10]:
def dataset_jsonl_transfer(origin_path, new_path):
    '''
    将原始数据集转换为大模型微调所需数据格式的新数据集
    '''
    messages =[]
    # 目的是微调下面四种情况 dict_keys(['text', 'entities', 'data_source'])
    match_names =['地点','人名','地理实体','组织']

    # 读取旧的JSONL文件
    with open(origin_path,'r')as file:
        for line in file:
            # 解析每一行的json数据
            data = json.loads(line)
            input_text = data['text']
            entities = data['entities']

            entity_sentence =''
            for entity in entities:
                entity_json = dict(entity)
                entity_text = entity_json['entity_text']
                entity_names = entity_json['entity_names']

                for name in entity_names:
                    if name in match_names:
                        entity_label = name
                        break

                entity_sentence += f'''{{'entity_text': '{entity_text}', 'entity_label': '{entity_label}'}}'''

            if entity_sentence == '':
                # print(1)
                entity_sentence ='没有找到任何实体'

            message ={
                'instruction':'''你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {'entity_text': '南京', 'entity_label': '地理实体'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出'没有找到任何实体'. ''',
                'input': f'文本:{input_text}',
                'output': entity_sentence,
            }

            messages.append(message)
            
            
    # print(messages)
    # 保存重构后的JSONL文件
    if new_path:
        with open(new_path,'w', encoding='utf-8')as file:
            for message in messages:
                file.write(json.dumps(message, ensure_ascii=False)+'\n')



In [11]:
# dataset_jsonl_transfer(argument.file_path, argument.train_file_path)

### 三 加载模型

In [2]:
model_id ='Qwen/Qwen2.5-0.5B'
model_dir ='./qwen'

In [3]:
# 在modelscope上下载Qwen模型到本地目录下
model_dir = snapshot_download(model_id, cache_dir='./', revision='master')

2024-09-23 16:32:48,560 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


In [4]:
# Transformers加载模型权重
tokenizer =AutoTokenizer.from_pretrained(model_dir, use_fast=False, trust_remote_code=True)

In [5]:
tokenizer

Qwen2Tokenizer(name_or_path='./Qwen/Qwen2___5-0___5B', vocab_size=151643, model_max_length=131072, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, 

In [27]:
# model =AutoModelForCausalLM.from_pretrained(model_dir, device_map='auto', torch_dtype=torch.bfloat16)
model =AutoModelForCausalLM.from_pretrained(model_dir, device_map='auto')
model.enable_input_require_grads()# 开启梯度检查点时，要执行该方法

  0%|          | 0/1768 [37:12<?, ?it/s]


In [8]:
def process_func(example):
    '''
    将数据集进行预处理
    '''
    MAX_LENGTH =384
    input_ids, attention_mask, labels =[],[],[]
    system_prompt ='''你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {'entity_text': '南京', 'entity_label': '地理实体'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出'没有找到任何实体'.'''

    instruction = tokenizer(
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{example['input']}<|im_end|>\n<|im_start|>assistant\n",
        add_special_tokens=False,
    )
    response = tokenizer(f"{example['output']}", add_special_tokens=False)
    input_ids = instruction['input_ids']+ response['input_ids']+[tokenizer.pad_token_id]
    attention_mask =(
        instruction['attention_mask']+ response['attention_mask']+[1]
    )
    labels =[-100]* len(instruction['input_ids'])+ response['input_ids']+[tokenizer.pad_token_id]
    if len(input_ids)> MAX_LENGTH:# 做一个截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {'input_ids': input_ids,'attention_mask': attention_mask,'labels': labels}



In [13]:
# 加载、处理数据集和测试集
# train_dataset_path ='ccfbdci.jsonl'
# train_jsonl_new_path ='ccf_train.jsonl'

# if not os.path.exists(train_jsonl_new_path):
#     dataset_jsonl_transfer(train_dataset_path, train_jsonl_new_path)

# 你不需要显式地将数据集移到 GPU 上，因为 Hugging Face Trainer 会自动处理这部分。
# 得到训练集
total_df = pd.read_json(argument.train_file_path, lines=True)
train_df = total_df[int(len(total_df)*0.1):]
train_ds =Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_func, remove_columns=train_ds.column_names)


Map: 100%|██████████| 14152/14152 [00:10<00:00, 1288.24 examples/s]


In [9]:
config =LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    inference_mode=False,# 训练模式
    r=8,# Lora 秩
    lora_alpha=32,# Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,# Dropout 比例
)

In [28]:
model = get_peft_model(model, config)

In [29]:
args =TrainingArguments(
    output_dir='./output/Qwen2-NER',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    logging_dir="./logs",  # 设置日志的输出目录
    logging_steps=10,
    num_train_epochs=1,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to='none',
    # fp16=True # 使用16位浮点数，适用于 GPU
)

# swanlab_callback =SwanLabCallback(
#     project='Qwen2-NER-fintune',
#     experiment_name='Qwen2-1.5B-Instruct',
#     description='使用通义千问Qwen2-1.5B-Instruct模型在NER数据集上微调，实现关键实体识别任务。',
#     config={
# 'model': model_id,
# 'model_dir': model_dir,
# 'dataset':'qgyd2021/chinese_ner_sft',
# },
# )



In [30]:

trainer =Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    # callbacks=[swanlab_callback],
)

trainer.train()

ValueError: fp16 mixed precision requires a GPU (not 'mps').

In [ ]:
def predict(messages, model, tokenizer):
    device ='cuda'
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors='pt').to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512
    )
    generated_ids =[
        output_ids[len(input_ids):]for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print(response)

    return response

In [ ]:

# 用测试集的随机20条，测试模型
# 得到测试集
test_df = total_df[:int(len(total_df)*0.1)].sample(n=20)

test_text_list =[]
for index, row in test_df.iterrows():
    instruction = row['instruction']
    input_value = row['input']

    messages =[
{'role':'system','content': f'{instruction}'},
{'role':'user','content': f'{input_value}'}
]

    response = predict(messages, model, tokenizer)
    messages.append({'role':'assistant','content': f'{response}'})
    result_text = f'{messages[0]}\n\n{messages[1]}\n\n{messages[2]}'
    test_text_list.append(swanlab.Text(result_text, caption=response))

swanlab.log({'Prediction': test_text_list})
swanlab.finish()

In [ ]:
['{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:联合国修改各会员国分摊联合国年度预算比例的截止时间就是２２号，而且联合国大会在圣诞节假期前夕比较容易通过修正案，如果这次联合国大会真的通过了削减美国分摊预算比例的修正案，美国将可以从现行分摊联合国每年２５％的预算比例降到２２％，至于分摊维持和平部队的比例也可望从３０％降到２６％到２７％。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'联合国\', \'entity_label\': \'组织\'}{\'entity_text\': \'联合国大会\', \'entity_label\': \'组织\'}{\'entity_text\': \'联合国大会\', \'entity_label\': \'组织\'}{\'entity_text\': \'联合国\', \'entity_label\': \'组织\'}{\'entity_text\': \'美国\', \'entity_label\': \'地理实体\'}{\'entity_text\': \'美国\', \'entity_label\': \'地理实体\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:从财政部的立场来讲，我们把一个客观的机制建立起来是最为重要的。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'财政部\', \'entity_label\': \'组织\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:在国际消息方面。\'}\n\n
{\'role\': \'assistant\', \'content\': \'没有找到任何实体\'}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:巴拉克说，他希望这一次与克林顿的会谈有助于终止流血行动。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'巴拉克\', \'entity_label\': \'人名\'}{\'entity_text\': \'克林顿\', \'entity_label\': \'人名\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:埃斯特拉达表示，这笔钱仍然原封不动的在银行中。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'埃斯特拉达\', \'entity_label\': \'人名\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:如果要，有香港人如果由陆路过去的真的很不，不方便。\'}\n\n
{\'role\': \'assistant\', \'content\': \'没有找到任何实体\'}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:陆委会副主委陈明通表示，开放大陆记者来台驻点采访是进一步的放宽，希望借由这一步的跨出，未来进一步透过协商能达成两岸正式互派记者常驻的目标。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'陆委会\', \'entity_label\': \'组织\'}{\'entity_text\': \'陈明通\', \'entity_label\': \'人名\'}{\'entity_text\': \'大陆\', \'entity_label\': \'地理实体\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:记者曾国华的报导。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'曾国华\', \'entity_label\': \'人名\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:继续是大陆地区以及国际的重要新闻。\'}\n\n
{\'role\': \'assistant\', \'content\': \'没有找到任何实体\'}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:更令人惊讶的是去年各界都认为米洛舍维奇一直全面掌控南斯拉夫政府各阶层，包括了传播媒体、安全部队、财经与司法部门，但是一等到塞尔维亚各地的工人与贝尔格勒的中产阶级知识分子同仇敌忾，清楚表明他们再也无法忍受独裁统治之后，米洛舍维奇看似坚固的政权结构就在一系之间溃败。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'米洛舍维奇\', \'entity_label\': \'人名\'}{\'entity_text\': \'南斯拉夫\', \'entity_label\': \'地理实体\'}{\'entity_text\': \'塞尔维亚\', \'entity_label\': \'地理实体\'}{\'entity_text\': \'贝尔格勒\', \'entity_label\': \'地理实体\'}{\'entity_text\': \'米洛舍维奇\', \'entity_label\': \'人名\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:领袖宣言由地主国苏丹？\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'苏丹\', \'entity_label\': \'地理实体\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:这边就珠海。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'珠海\', \'entity_label\': \'地理实体\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:ＥＭＰＴＹ\'}\n\n
{\'role\': \'assistant\', \'content\': \'没有找到任何实体\'}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:格瓦契夫２２号应邀在比利时参议院发表演说，他除了呼吁全面清除核子以及化学武器之外，并且呼吁欧洲的统合方面应该将俄罗斯涵概在内。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'格瓦契夫\', \'entity_label\': \'人名\'}{\'entity_text\': \'比利时\', \'entity_label\': \'地理实体\'}{\'entity_text\': \'参议院\', \'entity_label\': \'组织\'}{\'entity_text\': \'俄罗斯\', \'entity_label\': \'地理实体\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:联合国安理会１２月５号一致同意更新伊拉克的以石油换取食物方案。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'联合国安理会\', \'entity_label\': \'组织\'}{\'entity_text\': \'伊拉克\', \'entity_label\': \'地理实体\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:根据非正式的报告指出至少有６０名西藏难民在翻越喜玛拉亚山逃抵尼泊尔边界地区之后已经被交给中共公安。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'喜玛拉亚山\', \'entity_label\': \'地点\'}{\'entity_text\': \'尼泊尔\', \'entity_label\': \'地理实体\'}{\'entity_text\': \'中共公安\', \'entity_label\': \'组织\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:这架空中巴士客机之后安全降落在马尼拉爱奎诺国际机场。\'}\n\n
{\'role\': \'assistant\', \'content\': \'没有找到任何实体\'}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:毛高文说：相信未来在哥国国家广播电台运作之下，这套设备将可以发挥最大的功效，提供听众更好的服务。\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'毛高文\', \'entity_label\': \'人名\'}{\'entity_text\': \'哥国国家广播电台\', \'entity_label\': \'组织\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:对于出现这样的危机，不禁令人仔细的思考在过去的一连串危机中，给没有被打倒的，南斯拉夫联邦总统米洛舍维奇是否已经到了黔驴技穷？\'}\n\n
{\'role\': \'assistant\', \'content\': "{\'entity_text\': \'南斯拉夫联邦\', \'entity_label\': \'地理实体\'}{\'entity_text\': \'米洛舍维奇\', \'entity_label\': \'人名\'}"}', '{\'role\': \'system\', \'content\': "你是一个文本实体识别领域的专家，你需要从给定的句子中提取 地点; 人名; 地理实体; 组织 实体. 以 json 格式输出, 如 {\'entity_text\': \'南京\', \'entity_label\': \'地理实体\'} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出\'没有找到任何实体\'. "}\n\n
{\'role\': \'user\', \'content\': \'文本:他会不会发动最后一场战争？\'}\n\n
{\'role\': \'assistant\', \'content\': \'没有找到任何实体\'}']
